# KweriMoto: YAMNet baseline

**Frozen YAMNet features + a small three-class classifier** (healthy / engine_knock / misfire).

[Open in Colab](https://colab.research.google.com/github/SUUNARAYMOND/kwerimoto-dataset/blob/main/notebooks/yamnet_baseline.ipynb) · Dataset: [rayyyyy007/kwerimoto-engine-sounds](https://huggingface.co/datasets/rayyyyy007/kwerimoto-engine-sounds)

```
clip (16 kHz) → YAMNet (frozen) → 1,024 numbers per 0.96 s piece → combine per clip → small classifier → label
```

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load the dataset |
| 2 | Look at the data |
| 3 | Get YAMNet features |
| 4 | Sanity check: what does YAMNet hear? |
| 5 | Train the head |
| 6 | Test (once) |
| 7 | `real_test` |
| 8 | Check the mistakes |
| 9 | Save |

**Rules:** train on `train`, choose settings on `validation`, score `test` once at the end, never train or tune on `real_test`.

## 0. Setup

Fixed settings in one place. `SEED` makes random choices repeat on every run. On Colab, results are saved
to your Google Drive so they survive when the session ends.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

DATASET_ID = "rayyyyy007/kwerimoto-engine-sounds"
LABELS = ["healthy", "engine_knock", "misfire"]
SAMPLE_RATE = 16_000
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:  # on Colab: save to Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/kwerimoto/yamnet")
except ImportError:  # anywhere else: save next to the notebook
    SAVE_DIR = Path("outputs/yamnet")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Saving results to", SAVE_DIR)

## 1. Load the dataset

`snapshot_download` copies the dataset from Hugging Face (about 290 MB, once per session) and returns the folder.
`manifest.csv` has one row per clip: its file, label, split and source. We add `path`, the full path to each clip.

In [ ]:
from huggingface_hub import snapshot_download

root = Path(snapshot_download(DATASET_ID, repo_type="dataset"))
manifest = pd.read_csv(root / "manifest.csv", keep_default_na=False)
manifest["path"] = [str(root / f) for f in manifest["file_name"]]
print(len(manifest), "clips")
manifest.head(3)

## 2. Look at the data

Before training anything, check what we have: how many clips per label and split, and how long they are.

In [ ]:
pd.crosstab([manifest["label"], manifest["source_dataset"]], manifest["split"], margins=True, margins_name="total")[
    ["train", "validation", "test", "real_test", "total"]]

**Clip lengths.** YAMNet looks at 0.96 s pieces, so a clip shorter than 0.96 s gives a single piece that is
partly silence. Most of these short clips are synthetic knock, which is why we check later whether the model's
mistakes depend on length.

In [ ]:
display(manifest.groupby(["label", "source_dataset"])["duration_s"].describe()[["count", "min", "50%", "max"]].round(2))

short = manifest[manifest["duration_s"] < 0.96]
print(f"{len(short)} clips are shorter than one YAMNet piece (0.96 s):", short.groupby("label").size().to_dict())

fig, ax = plt.subplots(figsize=(8, 3))
for label in LABELS:
    ax.hist(manifest.loc[manifest["label"] == label, "duration_s"], bins=40, alpha=0.6, label=label)
ax.set_xlabel("clip length (s)"); ax.set_ylabel("clips"); ax.legend(); plt.show()

**Listen.** One random training clip per label.

In [ ]:
for label in LABELS:
    row = manifest[(manifest["split"] == "train") & (manifest["label"] == label)].sample(1, random_state=SEED).iloc[0]
    print(f"{label}: {row['file_name']} ({row['duration_s']} s, from {row['source_dataset']})")
    display(Audio(row["path"]))

## 3. Get YAMNet features

Load Google's YAMNet. It is **frozen**: we only use it, never train it.

For each clip YAMNet returns, per 0.96 s piece (a new piece every 0.48 s):
- **embedding**: 1,024 numbers describing how the piece sounds; this is what our classifier learns from;
- **scores**: how likely each of YAMNet's 521 sound types is (we use these in section 4).

Our labels are per clip, so we combine the pieces of each clip in two ways and later keep whichever works
better on `validation`:
- **mean**: the average piece, which describes the clip as a whole;
- **max**: the strongest value of each number over the pieces, which keeps short events (like a knock) that
  averaging could wash out.

This runs once (about 1 to 3 minutes on CPU) and is saved to Drive; later runs load the saved file.

In [ ]:
import subprocess
import sys

import tensorflow as tf
from tqdm.auto import tqdm

try:
    import kagglehub
except ImportError:  # Colab normally has it; install if missing
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

# Google's official YAMNet (formerly on TF Hub, now hosted on Kaggle)
yamnet = tf.saved_model.load(kagglehub.model_download("google/yamnet/tensorFlow2/yamnet"))
class_names = pd.read_csv(yamnet.class_map_path().numpy().decode())["display_name"].tolist()
print(len(class_names), "YAMNet sound types, e.g.", class_names[:3])


def yamnet_features(path):
    audio, sr = sf.read(path, dtype="float32")
    assert sr == SAMPLE_RATE and audio.ndim == 1, "YAMNet needs mono 16 kHz audio"
    scores, embeddings, _ = yamnet(audio)
    scores, embeddings = scores.numpy(), embeddings.numpy()
    return {"emb_mean": embeddings.mean(axis=0), "emb_max": embeddings.max(axis=0),
            "score_mean": scores.mean(axis=0), "score_max": scores.max(axis=0), "n_pieces": len(embeddings)}

In [ ]:
FEATURES = SAVE_DIR / "yamnet_features.npz"
file_names = manifest["file_name"].to_numpy(dtype=str)

if FEATURES.exists() and np.array_equal(np.load(FEATURES)["file_name"], file_names):
    feats = dict(np.load(FEATURES))
    print("Loaded saved features from", FEATURES)
else:
    rows = [yamnet_features(path) for path in tqdm(manifest["path"], desc="YAMNet")]
    feats = {key: np.stack([row[key] for row in rows]) for key in rows[0]}
    feats["file_name"] = file_names
    np.savez_compressed(FEATURES, **feats)
    print("Saved features to", FEATURES)

for key in ["emb_mean", "emb_max", "score_mean", "n_pieces"]:
    print(f"{key:10s} {feats[key].shape}")
manifest["n_pieces"] = feats["n_pieces"]
manifest.groupby("label")["n_pieces"].describe()[["min", "50%", "max"]]

## 4. Sanity check: what does YAMNet hear?

Before training, see what YAMNet itself recognises in our clips. Only `train` clips are used, so the other
splits stay untouched.

**Top sound types per label**: YAMNet's five highest average scores for each label. We hope to see engine and
vehicle sounds.

In [ ]:
train = (manifest["split"] == "train").to_numpy()
for label in LABELS:
    average = feats["score_mean"][train & (manifest["label"] == label).to_numpy()].mean(axis=0)
    top = np.argsort(average)[::-1][:5]
    print(f"{label:13s}", ", ".join(f"{class_names[i]} ({average[i]:.2f})" for i in top))

**YAMNet's own "Engine knocking" score.** YAMNet already knows a sound type called "Engine knocking". For each
clip we take its highest knock score over the pieces (0 = not heard, 1 = sure). If knock clips score clearly
higher than the others, YAMNet already hears knocking without any training from us. In section 6 we test this
properly as a no-training comparison for our classifier.

In [ ]:
KNOCK = class_names.index("Engine knocking")
manifest["yamnet_knock"] = feats["score_max"][:, KNOCK]
manifest[train].groupby(["label", "source_dataset"])["yamnet_knock"].describe()[["count", "25%", "50%", "75%"]].round(3)

## 5. Train the head

The only part we train: a small classifier that takes a clip's YAMNet numbers and outputs healthy / knock /
misfire. Every model learns from `train` only and is compared on `validation` only.

- **Macro-F1** is our score: the F1 score (balance of precision and recall) of each class, averaged so that
  misfire (108 clips) counts as much as knock (608). 1.0 is perfect.
- **Feature sets**: `mean`, `max`, or both side by side (`mean+max`, 2,048 numbers).
- **Scaling**: each of the numbers is shifted and stretched to a similar range, which helps training. The
  scaling is learned from `train` only.
- **Class weights** (`balanced`): a mistake on a rare class costs more, so the model cannot ignore misfire.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURE_SETS = {
    "mean": feats["emb_mean"],
    "max": feats["emb_max"],
    "mean+max": np.hstack([feats["emb_mean"], feats["emb_max"]]),
}
y = manifest["label"].map({label: i for i, label in enumerate(LABELS)}).to_numpy()
split = manifest["split"].to_numpy()
tr, va = split == "train", split == "validation"
print("train:", tr.sum(), "clips   validation:", va.sum(), "clips")

### 5a. Logistic regression

The simplest classifier: one weighted sum of the numbers per class. `C` controls how simple it must stay:
small `C` = simpler (less able to memorise), large `C` = freer.

If **train** F1 is much higher than **validation** F1, the model memorised the training clips instead of
learning something general (**overfitting**). We pick by validation.

In [ ]:
lr_results = []
for name, X in FEATURE_SETS.items():
    for C in [0.001, 0.01, 0.1, 1, 10]:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
        model.fit(X[tr], y[tr])
        lr_results.append({"features": name, "C": C,
                           "train_f1": f1_score(y[tr], model.predict(X[tr]), average="macro"),
                           "val_f1": f1_score(y[va], model.predict(X[va]), average="macro")})
lr_results = pd.DataFrame(lr_results)
best_lr = lr_results.loc[lr_results["val_f1"].idxmax()]
print(f"Best logistic regression: features={best_lr['features']}, C={best_lr['C']}, validation F1={best_lr['val_f1']:.3f}")
lr_results.pivot(index="C", columns="features", values=["train_f1", "val_f1"]).round(3)

### 5b. Small neural network

```
numbers (1,024 or 2,048) → 256 hidden units → ReLU → dropout → 3 scores
```

- **Hidden layer + ReLU**: lets the model combine numbers in ways a plain weighted sum cannot.
- **Dropout (0.5)**: during training, randomly switches off half the hidden units each step so the model
  cannot rely on memorising.
- **Loss** (cross-entropy with class weights): how wrong the model is; **Adam** adjusts the model to lower it.
- **Epoch**: one pass over all training clips, in **batches** of 32.
- **Early stopping**: after each epoch we check validation F1, keep the best version, and stop once it has not
  improved for 20 epochs.

In [ ]:
import copy

import torch
from torch import nn


def train_mlp(X, hidden=256, dropout=0.5, lr=1e-3, weight_decay=1e-4, max_epochs=300, patience=20, batch_size=32):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    scaler = StandardScaler().fit(X[tr])
    Xs = torch.tensor(scaler.transform(X), dtype=torch.float32)
    ys = torch.tensor(y)
    net = nn.Sequential(nn.Linear(X.shape[1], hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, len(LABELS)))
    class_weights = torch.tensor(tr.sum() / (len(LABELS) * np.bincount(y[tr], minlength=len(LABELS))), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)

    train_idx = np.flatnonzero(tr)
    best_f1, best_state, waited, history = -1.0, None, 0, []
    for epoch in range(max_epochs):
        net.train()
        losses = []
        for batch in np.array_split(rng.permutation(train_idx), int(np.ceil(len(train_idx) / batch_size))):
            optimizer.zero_grad()
            loss = loss_fn(net(Xs[batch]), ys[batch])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        net.eval()
        with torch.no_grad():
            val_f1 = f1_score(y[va], net(Xs[va]).argmax(1).numpy(), average="macro")
        history.append({"epoch": epoch + 1, "train_loss": np.mean(losses), "val_f1": val_f1})
        if val_f1 > best_f1:
            best_f1, best_state, waited = val_f1, copy.deepcopy(net.state_dict()), 0
        else:
            waited += 1
            if waited >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    return net, scaler, best_f1, pd.DataFrame(history)


mlp_runs = {name: train_mlp(X) for name, X in FEATURE_SETS.items()}
mlp_results = pd.DataFrame([{"features": name, "val_f1": run[2], "epochs": len(run[3])} for name, run in mlp_runs.items()])
best_mlp = mlp_results.loc[mlp_results["val_f1"].idxmax()]
print(f"Best neural network: features={best_mlp['features']}, validation F1={best_mlp['val_f1']:.3f}")
mlp_results.round(3)

**Training curve** of the best neural network: training loss should fall; validation F1 rises, then levels off.
The best epoch (dashed line) is the version we keep.

In [ ]:
history = mlp_runs[best_mlp["features"]][3]
fig, ax1 = plt.subplots(figsize=(8, 3))
ax1.plot(history["epoch"], history["train_loss"], color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("training loss", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(history["epoch"], history["val_f1"], color="tab:orange")
ax2.set_ylabel("validation macro-F1", color="tab:orange")
ax2.axvline(history.loc[history["val_f1"].idxmax(), "epoch"], color="gray", linestyle="--")
plt.show()

### 5c. Pick the winner

Rule (fixed before looking at any test results): the highest validation macro-F1 wins; if the two are within
0.01 of each other, logistic regression wins because it is simpler. The winner is the one model we test in
section 6.

In [ ]:
if best_mlp["val_f1"] > best_lr["val_f1"] + 0.01:
    net, scaler, _, _ = mlp_runs[best_mlp["features"]]
    chosen = {"model": "neural network", "features": best_mlp["features"], "val_f1": round(float(best_mlp["val_f1"]), 3)}

    def predict_proba(X):
        with torch.no_grad():
            return torch.softmax(net(torch.tensor(scaler.transform(X), dtype=torch.float32)), dim=1).numpy()
else:
    final_lr = make_pipeline(StandardScaler(), LogisticRegression(C=best_lr["C"], class_weight="balanced", max_iter=5000))
    final_lr.fit(FEATURE_SETS[best_lr["features"]][tr], y[tr])
    chosen = {"model": "logistic regression", "features": best_lr["features"], "C": float(best_lr["C"]), "val_f1": round(float(best_lr["val_f1"]), 3)}
    predict_proba = final_lr.predict_proba

X_chosen = FEATURE_SETS[chosen["features"]]
print("Chosen:", chosen)
print(classification_report(y[va], predict_proba(X_chosen[va]).argmax(1), target_names=LABELS, digits=3))

## 6. Test (once)

The chosen model meets `test` for the first time. **Do not go back and change the model because of these
numbers**: that would turn `test` into a second validation set and make the score too optimistic.

- **Precision** (per class): of the clips the model called this class, how many really were.
- **Recall** (per class): of the clips that really are this class, how many the model found.
- **Confusion matrix**: rows are the true label, columns what the model said. The diagonal is correct; every
  other cell shows which class gets mistaken for which.

In [ ]:
te, rt = split == "test", split == "real_test"
proba = predict_proba(X_chosen)
manifest["pred"] = np.array(LABELS)[proba.argmax(axis=1)]

test_f1 = f1_score(y[te], proba[te].argmax(axis=1), average="macro")
print(f"TEST macro-F1: {test_f1:.3f}   (validation was {chosen['val_f1']:.3f})\n")
print(classification_report(y[te], proba[te].argmax(axis=1), target_names=LABELS, digits=3))

confusion = pd.crosstab(manifest.loc[te, "label"], manifest.loc[te, "pred"]).reindex(index=LABELS, columns=LABELS, fill_value=0)
confusion.index.name, confusion.columns.name = "true", "predicted"
confusion

**For the report only: both classifiers on `test`.** The choice was already made on validation (section 5c)
and does not change; this just shows how the runner-up would have done.

In [ ]:
lr_model = make_pipeline(StandardScaler(), LogisticRegression(C=best_lr["C"], class_weight="balanced", max_iter=5000))
lr_model.fit(FEATURE_SETS[best_lr["features"]][tr], y[tr])
lr_test = lr_model.predict(FEATURE_SETS[best_lr["features"]][te])

mlp_net, mlp_scaler, _, _ = mlp_runs[best_mlp["features"]]
with torch.no_grad():
    mlp_test = mlp_net(torch.tensor(mlp_scaler.transform(FEATURE_SETS[best_mlp["features"]][te]), dtype=torch.float32)).argmax(1).numpy()

pd.DataFrame([
    {"model": "logistic regression", "features": best_lr["features"], "val_f1": best_lr["val_f1"],
     "test_f1": f1_score(y[te], lr_test, average="macro")},
    {"model": "neural network", "features": best_mlp["features"], "val_f1": best_mlp["val_f1"],
     "test_f1": f1_score(y[te], mlp_test, average="macro")},
]).assign(chosen=lambda d: d["model"] == chosen["model"]).round(3)

**No-training comparison: YAMNet's own "Engine knocking" score.** This only answers *knock or not*, so we
compare knock detection only. YAMNet alone needs a cut-off ("knock if the score is above X"); we choose the
cut-off that works best on `validation`, then apply it to `test`. If our trained model is not clearly better,
training added little.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

is_knock = (manifest["label"] == "engine_knock").to_numpy()
knock_score = manifest["yamnet_knock"].to_numpy()
cutoff = max(np.unique(knock_score[va]), key=lambda t: f1_score(is_knock[va], knock_score[va] >= t))


def knock_detection(said_knock, mask):
    precision, recall, f1, _ = precision_recall_fscore_support(is_knock[mask], said_knock[mask], average="binary")
    return {"precision": precision, "recall": recall, "F1": f1}


pd.DataFrame({
    "our model": knock_detection((manifest["pred"] == "engine_knock").to_numpy(), te),
    f"YAMNet alone (cut-off {cutoff:.2g})": knock_detection(knock_score >= cutoff, te),
}).T.round(3)

## 7. `real_test`

38 real knock recordings that YAMNet never trained on and that share no video with any of our training clips.
Every clip is knock, so the question is simply: **how many does the model catch?** Compare it with knock recall
on `test` above: a big drop means the model learned something about our training sources rather than knocking.

In [ ]:
caught = (manifest.loc[rt, "pred"] == "engine_knock").mean()
print(f"Our model calls {caught:.0%} of the {rt.sum()} real_test clips engine_knock")
print(f"YAMNet alone (same cut-off) catches {(knock_score[rt] >= cutoff).mean():.0%}")
print("\nWhat our model called them:", manifest.loc[rt, "pred"].value_counts().to_dict())

# real_test only has knock, so it cannot show false alarms; test can
not_knock = te & ~is_knock
print(f"\nFor context, false alarms on test (healthy and misfire clips called knock): "
      f"our model {(manifest.loc[not_knock, 'pred'] == 'engine_knock').mean():.0%}, "
      f"YAMNet alone {(knock_score[not_knock] >= cutoff).mean():.0%}")

## 8. Check the mistakes

Where does the model go wrong, and why? Here we only **look**: changing the model because of what we see would
turn `test` into a tuning set.

### 8a. Results per source

Each label comes from more than one source. For knock, `audioset` is real and `revix` is synthetic (a knock
mixed under a car recording). If one source is much worse, that is where the model struggles.

In [ ]:
seen = manifest[te | rt].assign(correct=lambda d: d["pred"] == d["label"])
by_source = pd.crosstab([seen["split"], seen["label"], seen["source_dataset"]], seen["pred"]).reindex(columns=LABELS, fill_value=0)
by_source.columns.name = "predicted"
by_source.insert(0, "clips", by_source.sum(axis=1))
by_source["share correct"] = seen.groupby(["split", "label", "source_dataset"])["correct"].mean()
by_source.round(2)

### 8b. Mistakes vs clip length

A clip shorter than 0.96 s gives YAMNet a single piece, partly silence. Are short clips wrong more often?
Most short clips are synthetic knock, so compare within each label. Some groups have only a few clips, so
one mistake moves their share a lot.

In [ ]:
length = pd.cut(manifest["duration_s"], [0, 0.96, 3, 6, 10.01], right=False,
                labels=["under 0.96 s (1 piece)", "0.96-3 s", "3-6 s", "6-10 s"])
seen_test = manifest[te].assign(length=length[te], correct=lambda d: d["pred"] == d["label"])
seen_test.groupby(["label", "length"], observed=True)["correct"].agg(clips="size", share_correct="mean").round(2)

### 8c. Listen to mistakes

For the two biggest mistakes (knock called healthy, healthy called knock), the clips the model was **surest**
about. Listen for: is the knock buried under the engine? Is the healthy clip really healthy? A confident
mistake can also be a wrong label in the source data.

In [ ]:
manifest["confidence"] = proba.max(axis=1)  # how sure the model was of its answer


def listen(true, said, n=3):
    wrong = manifest[te & (manifest["label"] == true).to_numpy() & (manifest["pred"] == said).to_numpy()]
    print(f"\n{true} called {said}: {len(wrong)} test clips. The {min(n, len(wrong))} the model was surest about:")
    for _, row in wrong.nlargest(n, "confidence").iterrows():
        print(f"{row['file_name']}  ({row['source_dataset']}, {row['duration_s']:.1f} s, {row['confidence']:.0%} sure)")
        display(Audio(row["path"]))


listen("engine_knock", "healthy")
listen("healthy", "engine_knock")

## 9. Save

Saved to `SAVE_DIR` (your Drive on Colab):

| File | What it is |
|---|---|
| `yamnet_head.pt` | the trained classifier and its scaling numbers (YAMNet itself is not saved; it is downloaded from Google) |
| `yamnet_results.json` | the scores from sections 5 to 8, for the report and for comparing with MFCC + XGBoost |
| `yamnet_predictions.csv` | every `test` and `real_test` clip with the model's answer and its probability for each label |

We then load the saved classifier back and check it gives the same answers, which proves the file works.

In [ ]:
if chosen["model"] == "neural network":
    model_file = SAVE_DIR / "yamnet_head.pt"
    torch.save({"state_dict": net.state_dict(), "in_features": X_chosen.shape[1], "hidden": net[0].out_features,
                "scaler_mean": torch.tensor(scaler.mean_), "scaler_scale": torch.tensor(scaler.scale_),
                "labels": LABELS, "features": chosen["features"]},  # mean+max = emb_mean then emb_max, side by side
               model_file)

    saved = torch.load(model_file)
    reloaded = nn.Sequential(nn.Linear(saved["in_features"], saved["hidden"]), nn.ReLU(), nn.Dropout(),
                             nn.Linear(saved["hidden"], len(saved["labels"])))
    reloaded.load_state_dict(saved["state_dict"])
    reloaded.eval()
    with torch.no_grad():
        x = (torch.tensor(X_chosen) - saved["scaler_mean"]) / saved["scaler_scale"]
        same = np.array_equal(reloaded(x.float()).argmax(1).numpy(), proba.argmax(axis=1))
else:
    import joblib

    model_file = SAVE_DIR / "yamnet_head.joblib"
    joblib.dump(final_lr, model_file)
    same = np.array_equal(joblib.load(model_file).predict_proba(X_chosen).argmax(1), proba.argmax(axis=1))

print("Saved", model_file)
print("Reloaded model gives the same answers:", same)

In [ ]:
import json

r3 = lambda v: round(float(v), 3)
report = classification_report(y[te], proba[te].argmax(axis=1), target_names=LABELS, output_dict=True)
results = {
    "model": f"frozen YAMNet + {chosen['model']}",
    "chosen": chosen,
    "dataset": DATASET_ID,
    "dataset_revision": root.name,  # the Hugging Face commit that was downloaded
    "seed": SEED,
    "validation_macro_f1": chosen["val_f1"],
    "test_macro_f1": r3(test_f1),
    "test_per_class": {label: {"precision": r3(report[label]["precision"]), "recall": r3(report[label]["recall"]),
                               "f1": r3(report[label]["f1-score"]), "clips": int(report[label]["support"])}
                       for label in LABELS},
    "test_confusion": {"rows_true_columns_predicted": LABELS, "counts": confusion.to_numpy().tolist()},
    "test_knock_false_alarms": r3((manifest.loc[not_knock, "pred"] == "engine_knock").mean()),
    "real_test_knock_recall": r3(caught),
    "share_correct_by_source": {" / ".join(key): r3(v) for key, v in by_source["share correct"].items()},
    "yamnet_alone": {"cutoff": float(cutoff),
                     "test_knock_detection": {k: r3(v) for k, v in knock_detection(knock_score >= cutoff, te).items()},
                     "real_test_knock_recall": r3((knock_score[rt] >= cutoff).mean())},
}
(SAVE_DIR / "yamnet_results.json").write_text(json.dumps(results, indent=2))

predictions = manifest.loc[te | rt, ["file_name", "split", "label", "source_dataset", "duration_s", "pred"]]
for i, label in enumerate(LABELS):
    predictions[f"p_{label}"] = proba[te | rt, i].round(3)
predictions.to_csv(SAVE_DIR / "yamnet_predictions.csv", index=False)

for f in sorted(SAVE_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1024:8.0f} KB")